<a href="https://colab.research.google.com/github/HazemmoAlsady/AWN_Graduation_Project/blob/main/classification_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [3]:
!kaggle datasets download -d kiva/data-science-for-good-kiva-crowdfunding

Dataset URL: https://www.kaggle.com/datasets/kiva/data-science-for-good-kiva-crowdfunding
License(s): CC0-1.0
100% 41.9M/41.9M [00:00<00:00, 92.1MB/s]



In [4]:
!unzip /content/data-science-for-good-kiva-crowdfunding.zip -d dataset

Archive:  /content/data-science-for-good-kiva-crowdfunding.zip
  inflating: dataset/kiva_loans.csv  
  inflating: dataset/kiva_mpi_region_locations.csv  
  inflating: dataset/loan_theme_ids.csv  
  inflating: dataset/loan_themes_by_region.csv  


In [5]:
!ls dataset

kiva_loans.csv		       loan_theme_ids.csv
kiva_mpi_region_locations.csv  loan_themes_by_region.csv


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import xgboost as xgb

In [7]:
import pandas as pd

df = pd.read_csv("/content/dataset/kiva_loans.csv")
df


,id,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,partner_id,posted_time,disbursed_time,funded_time,term_in_months,lender_count,tags,borrower_genders,repayment_interval,date
0,653051,300.0,300.0,Fruits & Vegetables,Food,"To buy seasonal, fresh fruits to sell.",PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:12:39+00:00,2013-12-17 08:00:00+00:00,2014-01-02 10:06:32+00:00,12.0,12,NaN,female,irregular,2014-01-01
1,653053,575.0,575.0,Rickshaw,Transportation,to repair and maintain the auto rickshaw used ...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:51:08+00:00,2013-12-17 08:00:00+00:00,2014-01-02 09:17:23+00:00,11.0,14,NaN,"female, female",irregular,2014-01-01
2,653068,150.0,150.0,Transportation,Transportation,To repair their old cycle-van and buy another ...,IN,India,Maynaguri,INR,334.0,2014-01-01 09:58:07+00:00,2013-12-17 08:00:00+00:00,2014-01-01 16:01:36+00:00,43.0,6,"user_favorite, user_favorite",female,bullet,2014-01-01
3,653063,200.0,200.0,Embroidery,Arts,to purchase an embroidery machine and a variet...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 08:03:11+00:00,2013-12-24 08:00:00+00:00,2014-01-01 13:00:00+00:00,11.0,8,NaN,female,irregular,2014-01-01
4,653084,400.0,400.0,Milk Sales,Food,to purchase one buffalo.,PK,Pakistan,Abdul Hakeem,PKR,245.0,2014-01-01 11:53:19+00:00,2013-12-17 08:00:00+00:00,2014-01-01 19:18:51+00:00,14.0,16,NaN,female,monthly,2014-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
671200,1340323,0.0,25.0,Livestock,Agriculture,"[True, u'para compara: cemento, arenya y ladri...",PY,Paraguay,Concepción,USD,58.0,2017-07-25 16:55:34+00:00,2017-07-25 07:00:00+00:00,NaN,13.0,0,NaN,female,monthly,2017-07-25
671201,1340316,25.0,25.0,Livestock,Agriculture,"[True, u'to start a turducken farm.'] - this l...",KE,Kenya,NaN,KES,138.0,2017-07-25 06:14:08+00:00,2017-07-24 07:00:00+00:00,2017-07-26 02:09:43+00:00,13.0,1,NaN,female,monthly,2017-07-25
671202,1340334,0.0,25.0,Games,Entertainment,NaN,KE,Kenya,NaN,KES,138.0,2017-07-26 00:02:07+00:00,2017-07-25 07:00:00+00:00,NaN,13.0,0,NaN,NaN,monthly,2017-07-26
671203,1340338,0.0,25.0,Livestock,Agriculture,"[True, u'to start a turducken farm.'] - this l...",KE,Kenya,NaN,KES,138.0,2017-07-26 06:12:55+00:00,2017-07-25 07:00:00+00:00,NaN,13.0,0,NaN,female,monthly,2017-07-26


In [8]:
df.isnull().sum()

,0
id,0
funded_amount,0
loan_amount,0
activity,0
sector,0
use,4232
country_code,8
country,0
region,56800
currency,0


In [9]:
df = df[["use", "sector"]]

df.dropna(inplace=True)

df.reset_index(drop=True, inplace=True)


/tmp/ipykernel_12925/1557571249.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [10]:
df["request_text"] = df["use"]

def map_label(sector):
    if sector in ["Food", "Agriculture", "Retail", "Transportation", "Services",
                  "Manufacturing", "Clothing", "Construction", "Wholesale"]:
        return "مالي"
    elif sector == "Education":
        return "تعليمي"
    elif sector == "Health":
        return "صحي"
    elif sector == "Housing":
        return "سكني"
    else:
        return "مالي"

df["label"] = df["sector"].apply(map_label)
df = df[["request_text", "label"]]

print("توزيع الفئات قبل التوازن:")
print(df["label"].value_counts())

/tmp/ipykernel_12925/522792405.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["request_text"] = df["use"]
/tmp/ipykernel_12925/522792405.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["label"] = df["sector"].apply(map_label)


توزيع الفئات قبل التوازن:
label
مالي      593393
سكني       33571
تعليمي     30837
صحي         9172
Name: count, dtype: int64


In [11]:
min_count = df["label"].value_counts().min()
df_balanced = df.groupby("label").apply(lambda x: x.sample(min_count, random_state=42)).reset_index(drop=True)

print("\nتوزيع الفئات بعد التوازن:")
print(df_balanced["label"].value_counts())


توزيع الفئات بعد التوازن:
label
تعليمي    9172
سكني      9172
صحي       9172
مالي      9172
Name: count, dtype: int64


/tmp/ipykernel_12925/98349364.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby("label").apply(lambda x: x.sample(min_count, random_state=42)).reset_index(drop=True)


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df_balanced["request_text"])


In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df_balanced["label"])


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [15]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [16]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred ))

              precision    recall  f1-score   support

           0       0.98      0.97      0.98      1835
           1       0.96      0.97      0.97      1834
           2       0.98      0.93      0.95      1835
           3       0.91      0.95      0.93      1834

    accuracy                           0.96      7338
   macro avg       0.96      0.96      0.96      7338
weighted avg       0.96      0.96      0.96      7338



In [18]:
test_text = ["Paying for medical treatment and checkups"]


test_vec = vectorizer.transform(test_text)
prediction = model.predict(test_vec)
print("Prediction:", le.inverse_transform(prediction))

Prediction: ['صحي']


In [19]:
test_data = [
    {"request_text": "I need money for medical treatment", "label": "صحي"},
    {"request_text": "Need help paying hospital bills", "label": "صحي"},
    {"request_text": "Support required for surgery", "label": "صحي"},

    {"request_text": "I want to pay school fees", "label": "تعليمي"},
    {"request_text": "Need help with university tuition", "label": "تعليمي"},
    {"request_text": "Support for children's education", "label": "تعليمي"},

    {"request_text": "I can't afford rent this month", "label": "سكني"},
    {"request_text": "Need help paying house rent", "label": "سكني"},
    {"request_text": "Support needed for home repair", "label": "سكني"},

    {"request_text": "I need money for food and groceries", "label": "مالي"},
    {"request_text": "Need support to buy food", "label": "مالي"},
    {"request_text": "Help required for daily expenses", "label": "مالي"},

    {"request_text": "I want to start a small business", "label": "مالي"},
    {"request_text": "Need money to buy tools for work", "label": "مالي"},
    {"request_text": "Support required for business startup", "label": "مالي"},
]

In [20]:
import pandas as pd

test_df = pd.DataFrame(test_data)

In [22]:
from sklearn.metrics import classification_report

X_test_text = test_df["request_text"]
y_true = test_df["label"]

# transform
X_test_vec = vectorizer.transform(X_test_text)

# predict
y_pred_encoded = model.predict(X_test_vec)
y_pred = le.inverse_transform(y_pred_encoded)

# evaluation
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

      تعليمي       0.75      1.00      0.86         3
        سكني       1.00      1.00      1.00         3
         صحي       1.00      1.00      1.00         3
        مالي       1.00      0.83      0.91         6

    accuracy                           0.93        15
   macro avg       0.94      0.96      0.94        15
weighted avg       0.95      0.93      0.94        15



In [23]:
for text, true, pred in zip(X_test_text, y_true, y_pred):
    print(f"Text: {text}")
    print(f"True: {true} | Pred: {pred}")
    print("-"*50)

Text: I need money for medical treatment
True: صحي | Pred: صحي
--------------------------------------------------
Text: Need help paying hospital bills
True: صحي | Pred: صحي
--------------------------------------------------
Text: Support required for surgery
True: صحي | Pred: صحي
--------------------------------------------------
Text: I want to pay school fees
True: تعليمي | Pred: تعليمي
--------------------------------------------------
Text: Need help with university tuition
True: تعليمي | Pred: تعليمي
--------------------------------------------------
Text: Support for children's education
True: تعليمي | Pred: تعليمي
--------------------------------------------------
Text: I can't afford rent this month
True: سكني | Pred: سكني
--------------------------------------------------
Text: Need help paying house rent
True: سكني | Pred: سكني
--------------------------------------------------
Text: Support needed for home repair
True: سكني | Pred: سكني
-------------------------------------

In [25]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.4 MB/s eta 0:00:00


In [26]:
from deep_translator import GoogleTranslator

def predict_arabic(text):
    # عربي → إنجليزي
    translated = GoogleTranslator(source='auto', target='en').translate(text)

    # prediction
    vec = vectorizer.transform([translated])
    pred = model.predict(vec)
    label = le.inverse_transform(pred)[0]

    return label

In [27]:
print(predict_arabic("محتاج عملية جراحية"))

صحي


In [28]:
test_data_ar = [
    # صحي
    "محتاج عملية جراحية عاجلة",
    "عايز علاج لابني",
    "مش قادر أدفع مصاريف المستشفى",
    "محتاج أدوية بشكل مستمر",
    "عايز أعمل كشف طبي",
    "بحاجة لدعم لعلاج مرض مزمن",
    "محتاج فلوس للعلاج",
    "عايز أعمل عملية قلب",

    # تعليمي
    "محتاج أدفع مصاريف المدرسة",
    "مش قادر أكمل تعليم ابني",
    "عايز فلوس للدراسة",
    "محتاج مصاريف جامعة",
    "عايز أشتري كتب دراسية",
    "مش لاقي فلوس للتعليم",
    "محتاج دعم للتعليم",
    "عايز أكمل دراستي",

    # سكني
    "مش قادر أدفع إيجار الشقة",
    "محتاج أصلح البيت",
    "البيت هيقع وعايز تصليح",
    "عايز مكان أعيش فيه",
    "محتاج فلوس للإيجار",
    "مش لاقي سكن مناسب",
    "البيت محتاج ترميم",
    "محتاج شقة للمعيشة",

    # مالي / عام
    "مش لاقي أكل لعيالي",
    "محتاج فلوس للأكل",
    "عايز أشتري أكل",
    "محتاج مصاريف يومية",
    "مش قادر أعيش",
    "محتاج دعم مالي",
    "عايز أفتح مشروع صغير",
    "محتاج فلوس للشغل",

    # حالات tricky 🔥
    "محتاج فلوس عشان أعيش",
    "ظروفي صعبة جدًا ومحتاج مساعدة",
    "مش لاقي شغل ومحتاج دعم",
    "عايز أساعد أسرتي",
    "محتاج أي نوع من المساعدة",
    "مش قادر أوفر احتياجات البيت",
    "عايز أبدأ من جديد",
    "محتاج حد يساعدني"
]

In [29]:
from deep_translator import GoogleTranslator

def predict_arabic_batch(texts):
    results = []

    for text in texts:
        # ترجمة
        translated = GoogleTranslator(source='auto', target='en').translate(text)

        # prediction
        vec = vectorizer.transform([translated])
        pred = model.predict(vec)
        label = le.inverse_transform(pred)[0]

        results.append({
            "arabic": text,
            "english": translated,
            "prediction": label
        })

    return results

In [30]:
results = predict_arabic_batch(test_data_ar)

for r in results:
    print(f"AR: {r['arabic']}")
    print(f"EN: {r['english']}")
    print(f"PRED: {r['prediction']}")
    print("-"*50)

AR: محتاج عملية جراحية عاجلة
EN: I need urgent surgery
PRED: صحي
--------------------------------------------------
AR: عايز علاج لابني
EN: I want treatment for my son
PRED: صحي
--------------------------------------------------
AR: مش قادر أدفع مصاريف المستشفى
EN: I cannot pay hospital expenses
PRED: صحي
--------------------------------------------------
AR: محتاج أدوية بشكل مستمر
EN: I need medications constantly
PRED: صحي
--------------------------------------------------
AR: عايز أعمل كشف طبي
EN: I want to do a medical examination
PRED: صحي
--------------------------------------------------
AR: بحاجة لدعم لعلاج مرض مزمن
EN: Need support to treat a chronic illness
PRED: صحي
--------------------------------------------------
AR: محتاج فلوس للعلاج
EN: I need money for treatment
PRED: صحي
--------------------------------------------------
AR: عايز أعمل عملية قلب
EN: I want to have a heart operation
PRED: صحي
--------------------------------------------------
AR: محتاج أدفع مصاريف المدر

In [31]:
import pandas as pd

df_results = pd.DataFrame(results)
df_results.head(20)

,arabic,english,prediction
0,محتاج عملية جراحية عاجلة,I need urgent surgery,صحي
1,عايز علاج لابني,I want treatment for my son,صحي
2,مش قادر أدفع مصاريف المستشفى,I cannot pay hospital expenses,صحي
3,محتاج أدوية بشكل مستمر,I need medications constantly,صحي
4,عايز أعمل كشف طبي,I want to do a medical examination,صحي
5,بحاجة لدعم لعلاج مرض مزمن,Need support to treat a chronic illness,صحي
6,محتاج فلوس للعلاج,I need money for treatment,صحي
7,عايز أعمل عملية قلب,I want to have a heart operation,صحي
8,محتاج أدفع مصاريف المدرسة,I need to pay school fees,تعليمي
9,مش قادر أكمل تعليم ابني,I cannot complete my son's education,تعليمي
